## Resumen Intro a Ciencias de Datos
### 1. Importar librerias


In [ ]:
#Primero importamos las librerias (todas las que nos podrían servir)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
import io, zipfile
from pathlib import Path
import jsons
import requests #(para Url)
import pyarrow.parquet as pq #(En caso de que se abra un archivo parquet)

### 2. Abrir archivos, extraer URL

In [ ]:
#Extraemos los datos desde un URL usando pandas
url = 'https://datos.gob.cl/dataset/c2969d8a-df82-4a6c-a1f8-e5eba36af6cf/resource/cbd329c6-9fe6-4dc1-91e3-a99689fd0254/download/pcma_20240917-oficio-4770_2013.xlsx'
#Leer datos dado los diferentes formatos, el skiprows es para ignorar filas del archivo
data = pd.read_excel(url,skiprows=9)
datos = pd.read_csv(url,skiprows=9)

#Otra forma de extraer un archivo desde su URL usando requests, stream es false para que descargue todo el body de una
respuesta = requests.get(url,stream=False)
#Se escribe el contenido del objeto en un archivo local en el computador
open("puntosBip.xlsx", "wb").write(respuesta.content)
#Luego se lee el archivo usando pandas
df = pd.read_excel("puntosBip.xlsx", engine="openpyxl",skiprows=9)
df = pd.read_parquet("puntosBip.xlsx", engine="o6penpyxl",skiprows=9)
df = pd.read_csv("puntosBip.xlsx", engine="openpyxl",skiprows=9)
#Extraer datos de una archivo zip
url = 'https://www.ine.gob.cl/docs/default-source/geodatos-abiertos/cartografia/censo-2017/siedu/shp/microdatos_manzana.zip?sfvrsn=972b0c54_3'
respuesta = requests.get(url,stream=True)
archivoZip = zipfile.ZipFile(io.BytesIo(respuesta.content))
archivoZip.extractall()
#Ahora abrimos el archivo
datos = pd.read_csv("Censo2017_16R_ManzanaEntidad_CSV/Censo2017_Manzanas.csv", delimiter=";")

#Extraemos datos de un Api
url = 'http://www.omdbapi.com/?t=adventure&y=1945'
respuesta = requests.get(url)
datos = respuesta.json()
#Pasamos los datos json a un dataframe
df = pd.json_normalize(respuesta.json(), record_path="people")

#Otra forma de pasar json a dataframe
rests = pd.json_normalize(df["businesses"], sep='_', record_path=['categories'], meta=['name','price','rating','review_count','distance',['coordinates','latitude'],['coordinates','longitude'],['location','address1']], errors='ignore')

#Leemos el URL que queremos abrir
taxi_data_URL = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2015-' + f'{RUT%12+1:02d}' + '.parquet'
#Abrimos el archivo
open("viajes.parquet","wb").write(requests.get(taxi_data_URL).content)
#Leemos el archivo parquet
viajes = pd.read_parquet("viajes.parquet")
#Dejar guardado en un DataFrame
viajes.to_csv("viajes.csv")

#Abrir URL de github (formato json)
data_url = "https://github.com/nyphilarchive/PerformanceHistory"
#Importamos requests 
respuesta = requests.get(data_url,stream=False)
open("conciertos.json","wb").write(respuesta.content)
data = json.load(open("data_2024\\filarmonica.json","r", encoding="UTF-8"))

#Cuando te pidan hacer un DataFrame con distintas columnas tenemos que empezar a crear ese dataframe
df = pd.json_normalize(data["programs"])
ny = pd.json_normalize(data["programs"], "concerts", ["programID","season"])
ny_works = pd.json_normalize(data["programs"], "works", ["programID"])
ny_works = ny_works[["ID", "composerName", "workTitle", "conductorName", "movement", "interval", "programID"]]
#Luego se hace el DataFrame final
nyphil_df = pd.merge(ny_works,ny, on="programID")
nyphil_df.head()


### 3. Acciones que se puedan hacer a partir del DataFrame

In [ ]:
#Mostrar las primeras n filas
data.head(11)#En caso hipotetico que pida las primeras 11 filas 

#Para ver la información de los tipos de datos de cada columna
data.info()

#Ver el total de cierta información del DataFrame
len(data)

#Para ver encontrar datos nulos tenemos dos opciones
data.isna() #Entrega un booleano indicando True si esque hay un valor nulo
data.notnull() #Devuelve True cuando el valor no es nulo y False si es nulo


#Para mostrar datos al azar de un DataFrame
data.sample(9)#En este caso mostramos 9 datos al azar

#Ver tipo de datos, si tenemos que un dato es de tipo de object mejor revisarlo, tenemos dos opciones para revisar:
#unique() y describe(). Si se tienen datos en especifico ya sea["M","N"] se dice que podrían ser variables categoricas
data["nombre_columna"].describe() #Resumen estadistico
data["nombre_columna"].unique() #Devuelve un array con los valores distintos que hay en la columna
#Si sale Tipo NaN quiere decir que no tienen datos 

#Eliminar columnas con datos nulos o datos que no sirvan
#Inplace modifica el Dataframe original elimiando la columna
data.drop(columns=["nombre_columna"],inplace=True)

#Para ver si hay datos duplicados ocupamos devuelve balor booleano, si sale True quiere decir que hay duplicados
duplicados = data.duplicated(keep=False)

#Para poder ver los valores duplicados creamos una fila especial para ver los duplicados
mostrar_duplicados = data[duplicados]

#Y si se quiere eliminar los duplicados, se pone el keep para ver si nos quedamos con el primero que sale en el DataFrame o el ultimo
data_limpio = data.drop_duplicates(keep="first")

#Para transformar datos a otro tipo de dato se utiliza astype puede ser a int64, float, string, category
data["nombre_columna"] = data["nombre_columna"].astype(".")

#Para poder ver valores mínimos y máximos de una columna especifico
data["nombre_columna"].describe()

#Para ver los datos de la columna en el cual se encuentra los datos mínimos
data.loc[data["nombre_columna"].idxmin()]

#Ahora para la columna donde se encuentra el valor máximo
data.loc[data["nombre_columna"].idxmax()]

#Si nos piden encontrar el número de veces donde más se repite tal dato
datos["nombre_columna"].value_counts()
#Si nos pidieran encontrar los top 10 que más se repiten sería así
top = data["nombre_columna"].value_counts()
top.head(10)

#Transformar a fecha
data["nombre_columna"] = pd.to_datetime(data["nombre_columna"])

#Cuando nos pidan poner condiciones al crear un nuevo dataframe de x datos
dato = 12
#Vamos poniendo las columnas que nos piden en el nuevo data de cierto dato que se nos este pidiendo con sus respectivas condiciones
nuevo_data = data[(data["nombre_columna"]==dato) & (dato["otra_columna"]>=1) & (data["otra_columnaa"]>=0.5) & (data["otra_columna_más"<=10])]

#Para resumir y comparar ciertos datos, podemos primero agrupar el dataframe de la columna que nos piden
#Agrupamos con groupby, calculamos para cada dato el promedio, minímo y maxímo
datos.groupby(by=["nombre_columna"]).agg({"otra_columna":["mean","min","max"]})

### 3. Creación de un gráfico


In [ ]:
#Gráfico de disperción
fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(111)
ax.plot(data["nombre_columna"],data["otra_columna"],".")
ax.set_xlabel("Información eje x")
ax.set_ylabel("Información eje y")
ax.set_title("Titulo")

#Otra forma de hacer grafico de dispercion 
fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(111)
ax.plot(sorted(dato.keys()), dato[sorted(dato.keys())],'.')
ax.set_xlabel("Información eje x")
ax.set_ylabel("Información eje y")
ax.set_title("Titulo")

#Grafico de violin
lista=[]
for i in range(1,data["nombre_columna"].max()+1):
    data_x = data[(data["nombre_columna"]==i)]
    lista.append(data["nombre_columna"])

fig = plt.figure(figsize=(10,10))
ax = fig.add_subplot(111)
ax.violinplot(lista, showmeans=True)
ax.set_xlabel("Información eje x")
ax.set_ylabel("Información eje y")
ax.set_title("Titulo")

#Gráfico de linea
fig = plt.figure(figsize=(10,10))
plt.plot(data.loc['nombre_dato']["nombre_columna"], 'x-', label='nombre_dato')
plt.plot(data.loc['nombre_dato']["nombre_columna"], 'x-', label='nombre_dato')
plt.legend()
ax.set_xlabel("Información eje x")
ax.set_ylabel("Información eje y")
plt.show()

#Gráfico de barra
plt.figure(figsize=(10,10))
#En este caso se están graficando los primeros 10 datos 
plt.bar(dato[:10].index, dato[:10].values, color = "red")
plt.title('Titulo')
ax.set_xlabel("Información eje x")
ax.set_ylabel("Información eje y")
plt.xticks(rotation=45)
plt.show()

### 4. Explicación de datos categoricos


Esta columna tiene datos cuyos valores están limitados a pocos posibles valores. Esto es justo lo que se necesita para el caso de variables categóricas.